# IELTS Listening — Generator QLoRA fine-tune (Kaggle)

Fine-tunes **Qwen2.5-7B-Instruct** with **QLoRA / SFT** to generate the
full doc contract from a spec:

> Blueprint -> Dialogue -> Audio Performance Instructions -> Questions ->
> Official Answers -> Accepted Variants -> Evaluation Metadata

(the *Core Generator Model* + *Training Objective* of
`AI IELTS Listening Exam Engine.md`).

**Run cell 0 first** to clone the repo (it has the SFT datasets), then
Run All.

### Before you run
1. Notebook settings: **Accelerator = GPU T4** (a *single* T4 — see below)
   and **Internet = On** (needed for the clone + the Unsloth install).
   Two traps: (a) **pick single *T4*, not *T4 x2***. Unsloth's free build
   trains on one GPU, and if two are visible it loads under an accelerate
   device_map dispatch that OOMs GPU 0 — the load cell now forces one GPU
   via `CUDA_VISIBLE_DEVICES=0` as a guard, but pick single T4 anyway. And
   (b) it needs a Turing+ card, so do **not** pick *P100*: it's Pascal
   (compute capability 6.0), Unsloth/Triton have no kernels for it, and it
   dies with `no kernel image is available for execution on the device`. So
   **a single T4 is the target** — 7B @ 8192 fits it comfortably (14B does
   not; see section 2).
2. **Get the data** — run **cell 0** to `git clone` this repo (default), or
   add `backend/data/datasets/generator_sft.jsonl` as a **Kaggle Dataset**
   named `ielts-listening-sft`. Cell 3 locates the jsonl either way.

The dataset is chat-format (`{messages:[system,user,assistant]}`); the
assistant turn is the exact JSON the backend already parses, so the
fine-tuned model is a drop-in for the hosted teacher.

## 0. Get the repo + datasets

In [15]:
# Cell 0 — pull this repo so the SFT datasets are on the Kaggle filesystem.
# This is a PRIVATE repo, so give Kaggle a token: Add-ons -> Secrets, add
# GITHUB_TOKEN = a PAT with read access to the repo (fine-grained 'Contents:
# read', or a classic token with the 'repo' scope).
# Alternative: skip cloning and add the 'ielts-listening-sft' Kaggle Dataset;
# the data cell below finds the jsonl either way.
import os
REPO_URL = "https://github.com/asiralabi/IELTS.git"
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{_tok}@")
except Exception:
    print("No GITHUB_TOKEN secret - trying anonymous clone "
          "(only works if the repo is public).")
if not os.path.isdir("ielts"):
    !git clone --depth 1 $REPO_URL ielts
!ls -lh ielts/backend/data/datasets/*.jsonl

No GITHUB_TOKEN secret - trying anonymous clone (only works if the repo is public).
-rw-r--r-- 1 root root 1.7M Jul 21 06:58 ielts/backend/data/datasets/cambridge_listening.jsonl
-rw-r--r-- 1 root root  11M Jul 21 06:58 ielts/backend/data/datasets/evaluator_sft.jsonl
-rw-r--r-- 1 root root 4.9M Jul 21 06:58 ielts/backend/data/datasets/generator_sft.jsonl


In [16]:
%%capture
# Unsloth gives ~2x faster QLoRA. Two Kaggle-specific gotchas: (1) the free
# Unsloth build trains on ONE GPU only, and (2) it needs a Turing-or-newer
# card — use a **T4** (compute capability 7.5). The P100 is Pascal (6.0) and
# has NO compiled Unsloth/Triton kernels -> 'no kernel image is available'.
# A 15 GB T4 fits Qwen2.5-7B @ 8192 comfortably (14B does NOT — it OOMs at
# train time, which is why the generator uses 7B; see its section 2).
# If this install ever breaks, follow the current Kaggle snippet at
# https://github.com/unslothai/unsloth (the API used below is stable).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load Qwen2.5-7B in 4-bit

In [17]:
import os
# Set BEFORE the torch/unsloth CUDA import. Lets the allocator grow
# segments instead of failing on a large contiguous request — cheap VRAM
# hygiene that reduces fragmentation-driven OOMs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pin to ONE GPU. Unsloth's free build trains on a single GPU anyway, but
# if TWO are visible (e.g. you picked 'T4 x2') it loads under an accelerate
# device_map dispatch whose per-forward hooks pile extra tensors onto GPU 0
# and OOM it. One visible GPU = clean single-device load. Set before import.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,   # QLoRA
)

==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 2. Attach LoRA adapters

The generator records are long — measured on the real corpus, the longest
is ~6.9k tokens (a ~2.8k-token system prompt + the full exam JSON) and 56
of 241 exceed 6144. So `MAX_SEQ_LEN` **must stay >= 7168** (8192 is the
safe value used here) or the assistant JSON gets truncated on the longest
records and the model learns to emit incomplete exams. Do **not** lower it
to save VRAM.

### Why 7B and not 14B
The doc's Core Generator is nominally 14B, but **Qwen2.5-14B @ 8192 will
not train on a Kaggle T4**: it loads fine, then OOMs at the first attention
forward inside `trainer.train()` (needs ~4 GiB, ~40 MB short of the 15 GB
card), and `expandable_segments` doesn't recover enough. The only other
free Kaggle card, P100, is Pascal and can't run Unsloth at all. So the
generator uses **Qwen2.5-7B**, which fits the T4 comfortably at the same
8192 context and keeps every training target intact. To train 14B you'd
need a >=24 GB Turing+ GPU (an A10/L4/3090 off-Kaggle) — then just set
`MODEL` back to `unsloth/Qwen2.5-14B-Instruct-bnb-4bit`.

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # long scripts -> save VRAM
    random_state=3407,
)

## 3. Load the SFT dataset

In [19]:
import glob, os
from datasets import load_dataset

# Resolves the SFT jsonl produced by backend/tools/build_dataset.py.
# Works whether you git-clone this repo into the notebook OR add it as a
# Kaggle Dataset named 'ielts-listening-sft'.
FILENAME = "generator_sft.jsonl"
CANDIDATES = [
    f"/kaggle/input/ielts-listening-sft/{FILENAME}",
    *glob.glob(f"/kaggle/**/backend/data/datasets/{FILENAME}", recursive=True),
    *glob.glob(f"**/backend/data/datasets/{FILENAME}", recursive=True),
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"{FILENAME} not found. Git-clone this repo (see cell 0) or add the "
        "Kaggle Dataset 'ielts-listening-sft'.")
print("Using dataset:", DATA_PATH)

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def to_text(row):
    # Each row is {"messages": [system, user, assistant]}; render with
    # the model's own chat template so training matches inference.
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset)
print(dataset[0]["text"][:600])

Using dataset: /kaggle/working/ielts/backend/data/datasets/generator_sft.jsonl
Dataset({
    features: ['text'],
    num_rows: 241
})
<|im_start|>system
You are an IELTS Listening test writer. Generate a complete, exam-authentic practice set built around a listening script.

Work in this order (the official design pipeline): first a Blueprint, then the Dialogue, then the Audio Performance Instructions, then the Questions, the Official Answers, the Accepted Variants, and finally the Evaluation Metadata.

Blueprint — REQUIRED `blueprint` object:
- Before writing the script, design the exam on paper. Add a top-level `blueprint` object:
  {
    "section": "<Part 1|Part 2|Part 3|Part 4>",
    "topic": "<the scenario in a few word


## 4. Train (response-only loss)

In [20]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=8192,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="qwen2.5-7b-ielts-generator-lora-checkpoints",
        report_to="none",
    ),
)

# Mask the prompt: only the assistant JSON contributes to the loss, so
# the model learns to PRODUCE exams, not to echo the spec/instructions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [21]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


ValueError: You can't train a model that has been loaded in 8-bit or 4-bit precision on a different device than the one you're training on. Make sure you loaded the model on the correct device using for example `device_map={'':torch.cuda.current_device()}` or `device_map={'':torch.xpu.current_device()}`

## 5. Save adapter + GGUF

In [ ]:
# 1) LoRA adapter only (tiny, ~100-300 MB) — load on top of the base.
model.save_pretrained("qwen2.5-7b-ielts-generator-lora")
tokenizer.save_pretrained("qwen2.5-7b-ielts-generator-lora")

# 2) GGUF (q4_k_m) for llama.cpp / Ollama serving on CPU or small GPU.
#    Merges + quantises; needs several GB of /kaggle/working disk.
model.save_pretrained_gguf("qwen2.5-7b-ielts-generator-gguf", tokenizer, quantization_method="q4_k_m")

# 3) (optional) merged 16-bit HF weights for vLLM (~14 GB for 7B) — uncomment
#    only if you have the disk / will push to the HF Hub.
# model.save_pretrained_merged("qwen2.5-7b-ielts-generator-lora-merged16", tokenizer, save_method="merged_16bit")

## 6. Serve it, and point the backend at it

**Option A — Ollama (local, CPU-friendly).** Download the GGUF, then:
```
# Modelfile
FROM ./qwen2.5-7b-ielts-generator-gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.4
PARAMETER num_ctx 8192
```
```
ollama create ielts-generator -f Modelfile
```
Then in `backend/.env`:
```
LLM_PROVIDER=ollama
OLLAMA_MODEL=ielts-generator
```

**Option B — vLLM (GPU, OpenAI-compatible).** Serve the merged weights
(or base + adapter) and set:
```
LLM_PROVIDER=openai
OPENAI_BASE_URL=http://<host>:8000/v1
OPENAI_MODEL=ielts-generator
OPENAI_API_KEY=dummy
```
No app code changes are needed — `app/llm/client.py` already speaks both.